<a href="https://colab.research.google.com/github/fawaztanigbola/Nigeria-ForeCasting-Model/blob/main/Southwest_Nigeria_Temperature_Project_Documentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Southwest Nigeria Temperature Forecasting
## Project Documentation: Baseline LSTM and Geospatial LSTM

**Study period:** 2015–2024  
**Study area:** Lagos, Ogun, Oyo and Osun states, Nigeria  
**Forecasting task:** Next-day minimum and maximum temperature prediction  
**Primary data source:** ERA5 reanalysis  
**Framework:** Python and PyTorch  
**Training environment:** Google Colab with an NVIDIA Tesla T4 GPU

---

## 1. Project overview

This project developed and evaluated two LSTM-based models for next-day minimum and maximum temperature forecasting across multiple geographical grid locations in Southwest Nigeria.

The work progressed in two main stages:

1. **Temperature-only baseline LSTM:** used the previous seven days of minimum and maximum temperature to predict the next day's values.
2. **Geospatial LSTM:** extended the temporal model with day-of-year seasonal features and static geographical information (latitude, longitude and elevation).

The project also compared model predictions with simpler forecasting references, tested alternative input-window lengths and model configurations, and examined whether static geographical features improved performance.

The aim was not simply to build a more complicated model. It was to test whether additional temporal and geographical information could provide useful forecasting improvements, while working within the available computing resources.

## 2. Why the study area was reduced

The initial ambition was to train and evaluate the models over a much larger geographical area covering Nigeria. During data preparation and experimentation, the size of a country-wide dataset created practical constraints in the available Google Colab environment.

The main constraints were:

- **Memory (RAM):** loading and processing a larger collection of geographical grid locations and daily observations required more memory.
- **Training time:** a much larger number of locations would increase the number of training examples and the time required to complete each training run.
- **Experimentation time:** model development requires repeated runs for validation, input-window comparisons and feature experiments. Training on the full country-wide dataset would make this iteration considerably slower with the resources available.
- **Available Colab resources:** the project had to remain within the practical memory, storage and compute capacity of the environment being used.

To make model development and experimentation feasible, the geographical scope was narrowed to Lagos and its surrounding region, represented in the project by **Lagos, Ogun, Oyo and Osun states**.

This was a resource-constrained scope decision, not a claim that Southwest Nigeria is the only region where the approach is relevant. The regional dataset still provided multiple grid locations and a ten-year time series, allowing the project to investigate temporal forecasting and the contribution of geographical features.

A future country-wide implementation could revisit the original ambition using more memory, faster or distributed compute, more efficient data pipelines, and appropriately designed spatial sampling or tiling.

**Important reporting note:** the project records support the qualitative explanation above. They do not provide a verified exact RAM peak, country-wide dataset size, or measured full-Nigeria training duration, so this documentation does not invent those figures.

## 3. Dataset and preparation

The notebooks report the following prepared dataset:

| Property | Description |
|---|---|
| Study period | 1 January 2015 – 31 December 2024 |
| Daily observations | 3,653 |
| Original regional grid | 28 × 24 = 672 cells |
| Usable geographical locations | 456 |
| Temperature variables | Minimum temperature (Tmin), maximum temperature (Tmax) |
| Temporal frequency | Daily |
| Target | Next-day Tmin and Tmax |

Grid cells with no usable temperature observations were excluded, leaving 456 locations for modelling.

### Data preparation

The geospatial notebook documents the following preparation steps:

- Daily ERA5 temperature data were prepared for the selected four-state region.
- Missing temperature values were interpolated along the time dimension.
- Elevation was extracted for the regional grid locations.
- Temperature values were scaled using a scaler fitted on the training period.
- Latitude, longitude and elevation were scaled separately.
- Day-of-year was transformed into sine and cosine features to represent annual seasonality cyclically.

The use of training-period statistics for scaling helps avoid using validation or test information to fit the scalers.

## 4. Model 1 — Temperature-only baseline LSTM

### Inputs and prediction

The baseline model used the previous **seven days** of:

- Minimum temperature (Tmin)
- Maximum temperature (Tmax)

It predicted the next day's Tmin and Tmax.

### Architecture and training configuration

| Component | Configuration |
|---|---|
| Model | Single-layer LSTM |
| Input features | 2 temperature variables |
| Sequence length | 7 days |
| Hidden size | 64 |
| Output | 2 values: next-day Tmin and Tmax |
| Optimizer | Adam |
| Learning rate | 0.001 |
| Loss | Mean Squared Error (MSE) |
| Maximum epochs | 20 |
| Early-stopping patience | 5 |
| Batch size | 64 |
| Hardware | NVIDIA Tesla T4 GPU |

The baseline notebook used a chronological 70% / 15% / 15% train-validation-test split. The model checkpoint with the lowest validation loss was retained.

### Best checkpoint

The lowest validation loss occurred at **epoch 1**, with a validation MSE loss of approximately **0.20159**. Validation loss increased in subsequent epochs, so early stopping ended training after epoch 6 and the best checkpoint was restored.

This is a useful model-selection result: the final model was not automatically taken from the last epoch. It was selected using validation performance.

## 5. Model 2 — Geospatial LSTM

### What changed?

The geospatial model added information intended to represent both seasonal timing and the location of each grid cell.

**Time-varying sequence inputs (4):**
- Scaled Tmin
- Scaled Tmax
- Day-of-year sine
- Day-of-year cosine

**Static geographical inputs (3):**
- Latitude
- Longitude
- Elevation

The LSTM processed the time-varying sequence. The three static geographical features were concatenated with the final LSTM hidden representation before the prediction head. They were **not** repeated as extra time-step channels.

### Architecture and training configuration

| Component | Configuration |
|---|---|
| Model | LSTM with a small prediction head |
| Sequence inputs | 4 |
| Static inputs | 3 (when enabled) |
| Hidden size | 64 |
| Selected sequence length | 7 days |
| Selected layers | 1 |
| Selected dropout | 0.0 |
| Prediction head | Linear → ReLU → Linear |
| Optimizer | Adam |
| Learning rate | 0.001 |
| Loss | MSE |
| Batch size | 1,024 |
| Gradient clipping | Maximum norm 1.0 |
| GPU acceleration | Tesla T4 with mixed precision |

### Chronological split

The geospatial notebook used fixed date boundaries:

- **Training:** 2015–2021
- **Validation:** 2022
- **Testing:** 2023–2024

The model configuration was selected using validation performance. The selected configuration was a 7-day input window, one LSTM layer and no dropout.

### Best checkpoint

The selected geospatial configuration achieved its best validation MSE at **epoch 13** (approximately **0.00393** in scaled units).

The best checkpoint was restored before the final validation and test predictions were evaluated. The geospatial validation loss is in scaled units and should not be directly compared with the baseline validation loss, because the notebooks use different scaling approaches.

## 6. Experiments conducted

### Experiment A — Input-window length

The geospatial model was tested with 7-, 14- and 30-day input windows. The 7-day window had the best validation average RMSE and was selected for the subsequent configuration experiment.

This indicates that, among the tested windows and under this setup, a longer history did not produce a better validation result.

### Experiment B — Number of layers and dropout

The notebook compared one- and two-layer LSTMs, with dropout values of 0.0 and 0.2. The one-layer model without dropout had the best validation average RMSE and was selected.

This provided an empirical basis for choosing a relatively compact architecture rather than assuming that a deeper or more heavily regularised model would perform better.

### Experiment C — Static geographical-feature ablation

The selected geospatial configuration was evaluated both with and without latitude, longitude and elevation.

| Configuration | Test average MAE | Test average RMSE |
|---|---:|---:|
| With latitude, longitude and elevation | 0.777945°C | 1.026826°C |
| Without static geographical features | 0.746635°C | 1.006270°C |

The static features slightly improved the average validation metrics, but that improvement did not carry through to the test set. On the test set, the version without static geographical features performed better on both average MAE and average RMSE.

**Interpretation:** adding geographical coordinates and elevation to this particular architecture did not produce a measurable test-set improvement. This does not establish that geographical information is unhelpful in general. It shows that these three static features, in the form and resolution tested here, were not sufficient to improve generalisation.

## 7. Test-set results

### Temperature-only baseline LSTM

| Target | MAE | RMSE |
|---|---:|---:|
| Tmin | 0.502°C | 0.690°C |
| Tmax | 0.988°C | 1.324°C |

### Geospatial LSTM (selected configuration)

| Target | MAE | RMSE |
|---|---:|---:|
| Tmin | 0.535°C | 0.707°C |
| Tmax | 1.021°C | 1.346°C |

MAE describes the average absolute difference between predictions and observed temperatures. RMSE also measures prediction error, while giving larger errors more influence.

Both models produced test errors below approximately 1°C MAE for Tmin. Tmax was more challenging, with MAE values close to 1°C. These are useful baseline results for a next-day, multi-location temperature forecasting project.

## 8. Forecast skill: improvement over reference models

MAE and RMSE are error measures; they are not accuracy percentages. A percentage that communicates how much error a model removes relative to a reference forecast can be calculated using a **forecast skill score**:

\[
\text{Skill (\%)} =
\left(1-\frac{\text{Model error}}{\text{Reference error}}\right)\times100
\]

A positive score means the model reduced error relative to the reference. A score of zero means it matched the reference, while a negative score means it performed worse.

For this project, persistence and day-of-year climatology are more meaningful reference forecasts than an undefined random guess:

- **Persistence:** tomorrow's temperature is predicted to be the same as the most recently observed temperature.
- **Day-of-year climatology:** the prediction uses the historical average for that day of the year, calculated from the training period.

A random guess for a continuous temperature value is not uniquely defined: its result depends on the distribution and range from which values are sampled. We therefore do not claim a measured improvement over a random predictor. Instead, we report skill against explicit, reproducible forecasting references.

In [1]:
# Forecast skill scores calculated from the test metrics documented above.
# Positive values indicate percentage error reduction versus the reference.

def skill_score(model_error, reference_error):
    return (1 - model_error / reference_error) * 100

comparisons = {
    "Baseline LSTM vs persistence": {
        "Tmin": {"MAE": (0.502, 0.576), "RMSE": (0.690, 0.778)},
        "Tmax": {"MAE": (0.988, 1.117), "RMSE": (1.324, 1.594)},
    },
    "Geospatial LSTM vs persistence": {
        "Tmin": {"MAE": (0.535, 0.583), "RMSE": (0.707, 0.779)},
        "Tmax": {"MAE": (1.021, 1.144), "RMSE": (1.346, 1.636)},
    },
    "Geospatial LSTM vs day-of-year climatology": {
        "Tmin": {"MAE": (0.535, 0.912), "RMSE": (0.707, 1.207)},
        "Tmax": {"MAE": (1.021, 1.378), "RMSE": (1.346, 1.731)},
    },
}

for comparison, targets in comparisons.items():
    print("\\n" + comparison)
    for target, metrics in targets.items():
        mae_skill = skill_score(*metrics["MAE"])
        rmse_skill = skill_score(*metrics["RMSE"])
        print(
            f"{target}: MAE skill = {mae_skill:.2f}% | "
            f"RMSE skill = {rmse_skill:.2f}%"
        )

\nBaseline LSTM vs persistence
Tmin: MAE skill = 12.85% | RMSE skill = 11.31%
Tmax: MAE skill = 11.55% | RMSE skill = 16.94%
\nGeospatial LSTM vs persistence
Tmin: MAE skill = 8.23% | RMSE skill = 9.24%
Tmax: MAE skill = 10.75% | RMSE skill = 17.73%
\nGeospatial LSTM vs day-of-year climatology
Tmin: MAE skill = 41.34% | RMSE skill = 41.43%
Tmax: MAE skill = 25.91% | RMSE skill = 22.24%


### Interpretation of the skill scores

The baseline LSTM reduced test error relative to persistence for both temperature targets:

- Tmin: approximately **12.85% MAE error reduction** and **11.31% RMSE error reduction**.
- Tmax: approximately **11.55% MAE error reduction** and **16.94% RMSE error reduction**.

On its own test period, the geospatial LSTM also reduced error relative to persistence:

- Tmin: approximately **8.23% MAE error reduction** and **9.24% RMSE error reduction**.
- Tmax: approximately **10.75% MAE error reduction** and **17.73% RMSE error reduction**.

The geospatial LSTM showed a larger improvement over day-of-year climatology:

- Tmin: approximately **41.34% MAE error reduction** and **41.43% RMSE error reduction**.
- Tmax: approximately **25.91% MAE error reduction** and **22.24% RMSE error reduction**.

These results show that the models learned useful forecasting patterns beyond the tested simple references. The skill scores are relative error reductions, not percentages of predictions that are exactly correct.

## 9. Comparing the two LSTMs fairly

The reported baseline LSTM test metrics are numerically lower than those of the geospatial LSTM:

| Target | Baseline MAE | Geospatial MAE | Baseline RMSE | Geospatial RMSE |
|---|---:|---:|---:|---:|
| Tmin | 0.502°C | 0.535°C | 0.690°C | 0.707°C |
| Tmax | 0.988°C | 1.021°C | 1.324°C | 1.346°C |

The baseline therefore has the lower reported test errors in this comparison. However, this should be treated as **indicative rather than a controlled head-to-head result**.

The baseline notebook uses a 70% / 15% / 15% chronological split, while the geospatial notebook uses 2015–2021 for training, 2022 for validation and 2023–2024 for testing. The notebooks also use different temperature-scaling approaches.

Consequently, the test periods and preprocessing are not confirmed to be identical. The metric values are still informative, but a definitive claim that one architecture is intrinsically better would require retraining both models with the same dates, data, preprocessing and evaluation procedure.

The most defensible conclusion is that the temperature-only baseline has the stronger reported test metrics, while the geospatial model demonstrates a working approach to incorporating seasonal and location information and performs better than its own reference forecasts.

## 10. Positive findings and project contributions

The project achieved several meaningful outcomes:

1. **Built two working LSTM forecasting pipelines** for next-day minimum and maximum temperature across hundreds of grid locations.
2. **Established a useful baseline:** the temperature-only LSTM outperformed persistence on both Tmin and Tmax.
3. **Extended the modelling approach:** the geospatial LSTM incorporated seasonal cyclical features and static geographical inputs.
4. **Compared against multiple references:** the geospatial model outperformed persistence and day-of-year climatology on its test period.
5. **Tested input history lengths:** the 7-day window was selected based on validation performance.
6. **Investigated model complexity:** one layer without dropout was selected based on validation results.
7. **Performed a feature ablation:** the experiment showed that latitude, longitude and elevation did not improve test performance in the current setup.
8. **Produced spatial diagnostics:** the geospatial notebook includes prediction-versus-actual plots, per-grid-cell error maps and an examination of error in relation to elevation.

These contributions demonstrate an end-to-end applied machine-learning workflow: data preparation, model design, training, validation, baseline comparison, feature experiments and interpretation.

## 11. Limitations and opportunities for improvement

The limitations below describe the boundaries of the current experiment. They do not invalidate the results.

### Geographic coverage and computing resources

The project was narrowed from the originally intended Nigeria-wide scope to four states because the larger dataset would have required more RAM and longer training time than was practical in the available Colab environment. A broader study could use higher-memory hardware, more efficient data loading, distributed processing or spatial tiling.

### Spatial detail and input features

The model used ERA5 grid-level temperature data and three static geographical variables. These inputs may not capture all local influences on temperature. Further work could investigate richer spatial and environmental predictors, provided suitable data are available and can be processed reliably.

### Geographical-feature result

Latitude, longitude and elevation did not improve the geospatial model's test metrics in the ablation experiment. The result suggests that simply appending static location features is not enough to guarantee better forecasts. Future work could test more informative spatial representations, neighbouring-grid context or additional meteorological variables.

### Missing-data treatment

The geospatial notebook reports temporal interpolation of missing temperature observations. Interpolation can make the time series more continuous, but it may smooth short-lived variations. Future work should quantify the effect of missing-data treatment and compare alternative strategies.

### Model generalisation and evaluation

The baseline training history showed its lowest validation loss at the first epoch, followed by rising validation loss. This indicates that the baseline began to overfit under the chosen configuration. Early stopping and checkpoint selection limited the impact of this behaviour.

The baseline and geospatial notebooks also use different chronological splits and scaling approaches. A same-split experiment is needed for a strict comparison between them.

### Forecasting scope

The current work predicts next-day temperatures at ERA5 grid locations. It should not be described as street-level forecasting or as a complete operational weather-forecasting system.

## 12. Recommended next steps

Prioritised improvements for a future iteration:

1. **Controlled model comparison:** retrain the temperature-only and geospatial models using exactly the same train, validation and test dates and the same preprocessing.
2. **Improve spatial representation:** test neighbouring grid-cell information or spatial architectures rather than relying only on static latitude, longitude and elevation.
3. **Add relevant environmental predictors:** investigate additional meteorological variables where data access and compute resources permit.
4. **Evaluate regional variation:** compare errors across grid locations, states and seasons to identify where forecasts are strongest and where they need improvement.
5. **Expand geographical coverage:** revisit Nigeria-wide modelling with a resource-efficient data pipeline and more capable compute.
6. **Strengthen uncertainty analysis:** examine prediction intervals or other uncertainty estimates before considering operational use.

These are proposed future investigations, not capabilities already demonstrated by the current models.

## 13. Overall conclusion

This project developed two LSTM-based approaches for next-day minimum and maximum temperature forecasting across 456 usable geographical grid locations in Southwest Nigeria, using daily data from 2015 to 2024.

The temperature-only baseline achieved MAE values of 0.502°C for Tmin and 0.988°C for Tmax, and reduced error relative to persistence. The geospatial LSTM extended the approach with seasonal and geographical information and outperformed both persistence and day-of-year climatology on its own test period.

The experiments also provided useful evidence about model design. A 7-day input window and a one-layer model without dropout were selected using validation performance. The static-feature ablation showed that latitude, longitude and elevation, as implemented here, did not improve test-set performance.

The project was scoped to Southwest Nigeria because a Nigeria-wide dataset would have placed demands on available Colab memory and training time that made iterative experimentation impractical. Within that constraint, the project produced two functioning forecasting models, quantitative comparisons with reference methods, spatial diagnostics and a clear set of next research directions.

**Overall status:** A completed regional forecasting study and a foundation for more extensive spatiotemporal temperature modelling. The results are encouraging, while the limitations and differences in evaluation protocols are documented transparently.